<a href="https://colab.research.google.com/github/SriVarshiniD/NIE-ML2/blob/main/ML2Lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# =====================================================
# 1. LOAD THE ONLINE SUPERVISED DATASET
# =====================================================

print("Loading Breast Cancer Wisconsin dataset...")

# Download the real dataset from scikit-learn
dataset = load_breast_cancer(as_frame=True)

X = dataset.data
y = dataset.target

# Class labels: 0 = malignant, 1 = benign
print("\nDataset loaded successfully!")
print("Dataset source:",
      "https://scikit-learn.org/stable/modules/generated/"
      "sklearn.datasets.load_breast_cancer.html")
print("Total samples:", X.shape[0])
print("Total features:", X.shape[1])
print("\nClass distribution:")
print(y.map({0: "malignant", 1: "benign"}).value_counts())

# =====================================================
# 2. SPLIT DATA INTO TRAINING AND TESTING SETS
# =====================================================

# Preserve original sample indices for per-sample reporting
sample_indices = np.arange(len(X))

X_train, X_test, y_train, y_test, idx_train, idx_test = (
    train_test_split(
        X,
        y,
        sample_indices,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# =====================================================
# 3. BUILD THE BAGGING CLASSIFIER
# =====================================================

n_estimators = 100

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=n_estimators,
    max_samples=1.0,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

print("\nBagging model trained successfully!")

# =====================================================
# 4. BUILD THE BOOSTING CLASSIFIER
# =====================================================

boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=42
    ),
    n_estimators=n_estimators,
    learning_rate=1.0,
    random_state=42
)

boosting.fit(X_train, y_train)
boosting_pred = boosting.predict(X_test)

print("Boosting (AdaBoost) model trained successfully!")

# =====================================================
# 5. EVALUATE MODEL PERFORMANCE
# =====================================================

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy (%)": accuracy_score(
            y_true, y_pred
        ) * 100,
        "Precision (%)": precision_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "F1-score (%)": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100
    }


results = pd.DataFrame([
    evaluate_model("Bagging", y_test, bagging_pred),
    evaluate_model("AdaBoost", y_test, boosting_pred)
])

print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(results.round(2).to_string(index=False))

# =====================================================
# 6. CONFUSION MATRICES
# =====================================================

print("\nBAGGING CONFUSION MATRIX")
print(confusion_matrix(y_test, bagging_pred))

print("\nBOOSTING CONFUSION MATRIX")
print(confusion_matrix(y_test, boosting_pred))

# =====================================================
# 7. CLASSIFICATION REPORTS
# =====================================================

print("\nBAGGING CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    bagging_pred,
    target_names=dataset.target_names,
    zero_division=0
))

print("\nBOOSTING CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    boosting_pred,
    target_names=dataset.target_names,
    zero_division=0
))

# =====================================================
# 8. BAGGING OUT-OF-BAG SCORE
# =====================================================

print("\n" + "=" * 60)
print("BAGGING OUT-OF-BAG PERFORMANCE")
print("=" * 60)

print(
    f"OOB Accuracy: {bagging.oob_score_ * 100:.2f}%"
)

# =====================================================
# 9. PER-SAMPLE REPETITION AND OOB ANALYSIS
# =====================================================

# Each array contains the training indices selected
# for one estimator's bootstrap sample.

bootstrap_samples = bagging.estimators_samples_

n_train = len(X_train)

# Statistics for each training sample
included_count = np.zeros(n_train, dtype=int)
total_draw_count = np.zeros(n_train, dtype=int)
duplicate_draw_count = np.zeros(n_train, dtype=int)
oob_count = np.zeros(n_train, dtype=int)

for selected_indices in bootstrap_samples:

    # Count how many times each sample was drawn
    counts = np.bincount(
        selected_indices,
        minlength=n_train
    )

    # Number of trees that included each sample
    included_count += (counts > 0)

    # Total number of draws across all trees
    total_draw_count += counts

    # Extra draws beyond the first occurrence
    duplicate_draw_count += np.maximum(counts - 1, 0)

    # A sample is OOB if it was not drawn for that tree
    oob_count += (counts == 0)

# Calculate percentages
inclusion_percentage = (
    included_count / n_estimators
) * 100

oob_percentage = (
    oob_count / n_estimators
) * 100

# Of all draws of a sample, what percentage were
# extra duplicate draws?
repeated_draw_percentage = (
    duplicate_draw_count
    / np.maximum(total_draw_count, 1)
) * 100

# Create a readable report
sample_report = pd.DataFrame({
    "Original_Row_Index": idx_train,
    "Sample_ID": X_train.index,
    "Actual_Class": y_train.map({
        0: "malignant",
        1: "benign"
    }).to_numpy(),

    "Bootstrap_Inclusion_Count": included_count,
    "Total_Draw_Count": total_draw_count,
    "Extra_Duplicate_Draws": duplicate_draw_count,

    "Included_In_Estimators_%": inclusion_percentage,
    "Repeated_Draws_%": repeated_draw_percentage,

    "OOB_Estimators_Count": oob_count,
    "OOB_%": oob_percentage
})

print("\n" + "=" * 60)
print("PER-SAMPLE BAGGING REPETITION AND OOB REPORT")
print("=" * 60)

print(sample_report.round(2).to_string(index=False))

# =====================================================
# 10. SUMMARY OF REPETITION AND OOB STATISTICS
# =====================================================

print("\n" + "=" * 60)
print("SUMMARY OF SAMPLING STATISTICS")
print("=" * 60)

print(
    "Average inclusion percentage:",
    round(inclusion_percentage.mean(), 2), "%"
)

print(
    "Average OOB percentage:",
    round(oob_percentage.mean(), 2), "%"
)

print(
    "Average repeated-draw percentage:",
    round(repeated_draw_percentage.mean(), 2), "%"
)

print(
    "Training samples with at least one OOB estimator:",
    int((oob_count > 0).sum()),
    "out of",
    n_train
)

# =====================================================
# 11. IDENTIFY THE BETTER TEST-SET MODEL
# =====================================================

best_model = results.loc[
    results["F1-score (%)"].idxmax()
]

print("\n" + "=" * 60)
print("BEST MODEL BASED ON WEIGHTED F1-SCORE")
print("=" * 60)

print("Model:", best_model["Model"])
print("Accuracy:", round(best_model["Accuracy (%)"], 2), "%")
print("Weighted F1-score:", round(best_model["F1-score (%)"], 2), "%")

# =====================================================
# 12. SAVE REPORTS TO CSV FILES
# =====================================================

results.to_csv(
    "model_performance.csv",
    index=False
)

sample_report.to_csv(
    "bagging_sample_analysis.csv",
    index=False
)

print("\n" + "=" * 60)
print("REPORTS SAVED SUCCESSFULLY")
print("=" * 60)

print("1. model_performance.csv")
print("2. bagging_sample_analysis.csv")


# =====================================================
# 13. FIND EPSILON FOR BAGGING MODEL CONVERGENCE
# =====================================================

print("\n" + "=" * 60)
print("EPSILON / CONVERGENCE ANALYSIS")
print("=" * 60)

estimator_values = list(range(10, 201, 10))
epsilon_threshold = 0.001  # 0.1 percentage points in accuracy
consecutive_steps = 3

accuracy_values = []

for n in estimator_values:

    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=n,
        max_samples=1.0,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    accuracy_values.append(accuracy)

epsilon_results = pd.DataFrame({
    "Estimators": estimator_values,
    "Accuracy": accuracy_values
})

epsilon_results["Accuracy_Change"] = (
    epsilon_results["Accuracy"].diff().abs()
)

print("\nAccuracy convergence table:")
print(epsilon_results.round(5).to_string(index=False))

# Find the first point with sufficiently small changes
# for several consecutive estimator increments.
epsilon_results["Below_Epsilon"] = (
    epsilon_results["Accuracy_Change"] < epsilon_threshold
)

estimated_stable_estimators = None

for i in range(consecutive_steps, len(epsilon_results)):

    recent_changes = epsilon_results["Below_Epsilon"].iloc[
        i - consecutive_steps + 1:i + 1
    ]

    if recent_changes.all():
        estimated_stable_estimators = int(
            epsilon_results.iloc[i]["Estimators"]
        )
        break

print("\nEpsilon threshold:", epsilon_threshold)
print("Equivalent accuracy change:", epsilon_threshold * 100,
      "percentage points")

if estimated_stable_estimators is not None:
    print("Estimated convergence point:",
          estimated_stable_estimators, "estimators")
    print("Accuracy changes stayed below epsilon for",
          consecutive_steps, "consecutive increments.")
else:
    print("No convergence point found within the tested range.")

epsilon_results.to_csv(
    "epsilon_convergence_analysis.csv",
    index=False
)

print("Saved: epsilon_convergence_analysis.csv")


Loading Breast Cancer Wisconsin dataset...

Dataset loaded successfully!
Dataset source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
Total samples: 569
Total features: 30

Class distribution:
target
benign       357
malignant    212
Name: count, dtype: int64

Training samples: 455
Testing samples: 114

Bagging model trained successfully!
Boosting (AdaBoost) model trained successfully!

MODEL PERFORMANCE COMPARISON
   Model  Accuracy (%)  Precision (%)  Recall (%)  F1-score (%)
 Bagging         93.86          93.84       93.86         93.84
AdaBoost         95.61          95.69       95.61         95.58

BAGGING CONFUSION MATRIX
[[38  4]
 [ 3 69]]

BOOSTING CONFUSION MATRIX
[[38  4]
 [ 1 71]]

BAGGING CLASSIFICATION REPORT
              precision    recall  f1-score   support

   malignant       0.93      0.90      0.92        42
      benign       0.95      0.96      0.95        72

    accuracy                           0.94       114
   